# Class 26: The ReAct Loop

এই notebook-এ আমরা ধাপে ধাপে একটা ReAct loop নিজের হাতে বানাব: Thought, Action, Observation। প্রথমে একটা scripted fake LLM দিয়ে loop বুঝব (Ollama লাগবে না), শেষে চাইলে real Ollama model চালাব।

## 1. Setup

Project folder-এর code আমরা সরাসরি import করব, যাতে lecture আর project একই loop ব্যবহার করে।

In [ ]:
import sys, json, tempfile
from pathlib import Path

PROJECT = Path("../Class 2 Project").resolve()
sys.path.insert(0, str(PROJECT))

from src.tools import TravelTools, ToolError
from src.react_agent import run_agent, parse_step

tmp = tempfile.mkdtemp()
tools = TravelTools(str(PROJECT / "data" / "flights.json"), str(Path(tmp) / "bookings.json"))
print(len(tools.flights), "flights loaded")

## 2. Tool দুটো আলাদাভাবে

LLM tool চালায় না, আমাদের code চালায়। তাই আগে tool দুটো নিজে চালিয়ে দেখি। লক্ষ্য করুন ভুল input-এ `ToolError` আসে, crash নয়।

In [ ]:
for f in tools.search_flights("DAC", "CXB", date="2026-11-05"):
    print(f["id"], f["departs"], f["price_bdt"], "seats:", f["seats_left"])

try:
    tools.book_flight("BG102", "Rahim")   # sold out
except ToolError as e:
    print("ToolError:", e)

## 3. Scripted fake LLM দিয়ে loop

Fake LLM হলো এমন একটা function যে আগে থেকে লেখা reply একটা একটা করে ফেরত দেয়। এতে loop-এর আচরণ হুবহু পুনরাবৃত্তি করা যায়। নিচে sold-out ফ্লাইট বুক করার চেষ্টা, error পড়ে শুধরে নেওয়া, তারপর সফল booking।

In [ ]:
class FakeLLM:
    def __init__(self, replies): self.replies = list(replies)
    def __call__(self, messages): return json.dumps(self.replies.pop(0))

def act(name, **kw): return {"thought": f"calling {name}", "action": name, "action_input": kw}

def show(step):
    print(f"[{step.n}] {step.action or 'FINAL'} {step.action_input or ''}\n     -> {step.observation[:110]}")

llm = FakeLLM([
    act("search_flights", origin="DAC", destination="CXB", date="2026-11-05"),
    act("book_flight", flight_id="BG102", passenger_name="Rahim"),   # sold out, ভুল
    act("book_flight", flight_id="BG101", passenger_name="Rahim"),   # শুধরানো
    {"thought": "done", "final_answer": "Booked BG101 for Rahim"},
])
res = run_agent("Book DAC to CXB on 2026-11-05 for Rahim", tools, llm, on_step=show)
print(res.status, "|", res.answer)

## 4. Failure handling: চারটা লাগাম

প্রতিটা লাগাম আলাদা করে ঘটিয়ে দেখি। প্রতিবার আলাদা `TravelTools` নিই যাতে booking state মিশে না যায়।

In [ ]:
def fresh():
    return TravelTools(str(PROJECT / "data" / "flights.json"), str(Path(tempfile.mkdtemp()) / "b.json"))

# ১. ভাঙা output: exception নয়, Observation
class RawLLM:
    def __init__(self, replies): self.replies = list(replies)
    def __call__(self, m): return self.replies.pop(0)
r = run_agent("x", fresh(), RawLLM(["sorry, I think...", json.dumps({"thought":"","final_answer":"ok"})]), on_step=show)
print("১.", r.status)

# ২. max_steps: প্রতিবার আলাদা input, তাই loop detection ধরে না
llm = FakeLLM([act("search_flights", origin="DAC", destination="CXB", max_price=9000+i) for i in range(5)])
print("২.", run_agent("x", fresh(), llm, max_steps=3).status)

# ৩. loop detection: হুবহু একই action
same = act("search_flights", origin="DAC", destination="CXB")
print("৩.", run_agent("x", fresh(), FakeLLM([same]*5), max_repeat_actions=2).status)

# ৪. LLM বন্ধ: retry শেষে llm_failed
def dead(messages): raise ConnectionError("ollama down")
print("৪.", run_agent("x", fresh(), dead, max_llm_retries=2).status)

## 5. Real Ollama (ঐচ্ছিক)

Ollama চালু না থাকলে নিচের cell crash করবে না, `llm_failed` দেখাবে।

### Model Reference Table

| Model | Use Case | Advantage | Limitation |
| --- | --- | --- | --- |
| **`llama3.1:8b`** (default) | Multi-step reasoning + tool use | JSON ও instruction ভালো মানে | ~16GB RAM লাগে |
| `mistral:7b` | দ্রুত general-purpose agent | কম latency | Format মাঝে মাঝে ভাঙে |
| `llama3.2:3b` | Low-RAM মেশিন, সহজ task | ছোট ও দ্রুত | বেশি step-এ হারায়, loop-এ আটকায় |
| Cloud (GPT-4 / Claude) | সবচেয়ে নির্ভরযোগ্য agent | Planning সেরা | Local না, খরচ আছে, data বাইরে যায় |

In [ ]:
import functools
from src.llm_client import ollama_chat

TASK = "Book the cheapest DAC to CXB flight on 2026-11-05 for Rahim"
for model in ["llama3.1:8b", "mistral:7b", "llama3.2:3b"]:
    print("=====", model)
    r = run_agent(TASK, fresh(), functools.partial(ollama_chat, model=model), on_step=show)
    print(r.status, "|", r.answer[:150])

## 6. Brain Teasers

1. তিন model-এর ফলাফল পাশাপাশি table-এ লিখুন: status, step সংখ্যা, প্রথম ভুল কোথায়।
2. কোনো ফ্লাইট নেই এমন task দিন। Agent সৎ উত্তর দেয় কি?
3. Loop detection ফাঁকি দেওয়া একটা fake script লিখুন, কোন লাগাম থামায় দেখুন।